In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, sys

In [3]:
import pandas as pd

#df = pd.read_csv("updated_df.csv")
transformed_metrics = pd.read_csv("transformed_metrics.csv")
df = transformed_metrics
df_varian = df[df["MachineType"] == "TrueBeam"]
df_elekta = df[df["MachineType"] == "Agility"]


In [4]:
df.columns

Index(['QADate', 'Unit', 'Energy', 'MU', 'DPF', 'ADPass', 'RDPass', 'AD', 'RD',
       'D4DD', 'D4DTA', 'D4Gamma', 'Area', 'Circumference', 'MCS', 'Span',
       'CoA', 'MachineType', 'Mu_per_dpf', 'log_Area', 'square_Area',
       'sqrt_Area', 'inv_Area', 'log_Circumference', 'square_Circumference',
       'sqrt_Circumference', 'inv_Circumference', 'log_MCS', 'square_MCS',
       'sqrt_MCS', 'inv_MCS', 'log_Span', 'square_Span', 'sqrt_Span',
       'inv_Span', 'log_CoA', 'square_CoA', 'sqrt_CoA', 'inv_CoA', 'log_MU',
       'square_MU', 'sqrt_MU', 'inv_MU', 'log_Mu_per_dpf', 'square_Mu_per_dpf',
       'sqrt_Mu_per_dpf', 'inv_Mu_per_dpf'],
      dtype='object')

In [5]:
X = df[['inv_Area', 'Circumference', 'log_MU', "square_Span" ,'inv_CoA','inv_MCS','log_Mu_per_dpf']]
y = df["AD"]

In [10]:
X_varian = df_varian[['inv_Area', 'Circumference', 'log_MU', "square_Span" ,'inv_CoA','inv_MCS','log_Mu_per_dpf']]
y_varian = df_varian["AD"]

In [14]:
X_elekta = df_elekta[['inv_Area', 'Circumference', 'log_MU', "square_Span" ,'inv_CoA','inv_MCS','log_Mu_per_dpf']]
y_elekta = df_elekta["AD"]

In [ ]:
# X_inv = transformed_metrics[['inv_Area', 'inv_Circumference', 'inv_MU', "inv_Span" ,'inv_CoA','inv_MCS','inv_Mu_per_dpf']]
# y_transformed = transformed_metrics["AD"]

In [ ]:
# scale = StandardScaler()
# X_scaled = scale.fit_transform(X_inv)
# X_inv_train, X_inv_test, y_inv_train, y_inv_test = train_test_split(X_scaled, y_transformed, test_size=0.25, random_state=42)

In [12]:

# # scale X
scale = StandardScaler()
scaled_X = scale.fit_transform(X)

# # train test split
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(scaled_X, y, test_size = 0.25, random_state = 4)


In [15]:
scale = StandardScaler()
scaled_X_e = scale.fit_transform(X_elekta)
X_train_E, X_test_E, y_train_E, y_test_E = train_test_split(X_elekta, y_elekta, test_size=0.25, random_state=42)

In [16]:
scale = StandardScaler()
scaled_X_v = scale.fit_transform(X_varian)
X_train_V, X_test_V, y_train_V, y_test_V = train_test_split(X_varian, y_varian, test_size=0.25, random_state=42)

RandomForestRegressor(max_depth=9, n_estimators=10)

In [18]:


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# Each dataset: (X_train, X_test, y_train, y_test)
datasets = {
    "Full":   (X_train_full, X_test_full, y_train_full, y_test_full),
    "Elekta": (X_train_E, X_test_E, y_train_E, y_test_E),
    "Varian": (X_train_V, X_test_V, y_train_V, y_test_V)
    #"Inverted": (X_inv_train, X_inv_test, y_inv_train, y_inv_test)
}

param_grid = {
    'n_estimators': [5, 10, 15, 20],
    'max_depth': [2, 5, 7, 9]
}

results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== Running Random Forest for {name} dataset ===")

    # Grid search for hyperparameter tuning
    grid = GridSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid=param_grid,
        cv=10,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_n = grid.best_params_["n_estimators"]
    best_depth = grid.best_params_["max_depth"]
    print(f"Best params for {name}: n_estimators={best_n}, max_depth={best_depth}")

    # Fit final model
    final = RandomForestRegressor(
        n_estimators=best_n,
        max_depth=best_depth,
        random_state=42
    )
    final.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = final.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Performance on {name} test set:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE : {mae:.4f}")
    print(f"  R²  : {r2:.4f}")

    # Store results
    results.append({
        "Dataset": name,
        "Best_n_estimators": best_n,
        "Best_max_depth": best_depth,
        "Best_CV_Score": grid.best_score_,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

# Summary table
results_df = pd.DataFrame(results)
print("\n=== Summary of Model Results ===")
print(results_df.round(4))



=== Running Random Forest for Full dataset ===
Best params for Full: n_estimators=20, max_depth=9
Performance on Full test set:
  RMSE: 5.3390
  MAE : 4.0466
  R²  : 0.5156

=== Running Random Forest for Elekta dataset ===
Best params for Elekta: n_estimators=15, max_depth=9
Performance on Elekta test set:
  RMSE: 5.0138
  MAE : 3.8458
  R²  : 0.5419

=== Running Random Forest for Varian dataset ===
Best params for Varian: n_estimators=20, max_depth=9
Performance on Varian test set:
  RMSE: 4.5502
  MAE : 3.3618
  R²  : 0.5153

=== Summary of Model Results ===
  Dataset  Best_n_estimators  Best_max_depth  Best_CV_Score    RMSE     MAE  \
0    Full                 20               9         0.4635  5.3390  4.0466   
1  Elekta                 15               9         0.4461  5.0138  3.8458   
2  Varian                 20               9         0.2967  4.5502  3.3618   

       R2  
0  0.5156  
1  0.5419  
2  0.5153  


In [19]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd


# Parameter grid for tuning
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1]
}

results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== Running XGBoost for {name} dataset ===")

    # Define base model
    xgbr = xgb.XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    # Grid search with 10-fold CV
    grid = GridSearchCV(
        xgbr,
        param_grid=param_grid,
        cv=10,
        scoring="r2",   # use R² as CV metric
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    # Best parameters
    best_params = grid.best_params_
    print(f"Best params for {name}: {best_params}")

    # Fit final model with best params
    best_xgbr = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=42
    )
    best_xgbr.fit(X_train, y_train)

    # Predict on test set
    y_pred = best_xgbr.predict(X_test)

    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Performance on {name} test set:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE : {mae:.4f}")
    print(f"  R²  : {r2:.4f}")

    # Store results
    results.append({
        "Dataset": name,
        "Best_n_estimators": best_params["n_estimators"],
        "Best_max_depth": best_params["max_depth"],
        "Best_learning_rate": best_params["learning_rate"],
        "Best_CV_R2": grid.best_score_,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

# Summary table
results_df = pd.DataFrame(results)
print("\n=== XGBoost Results Summary ===")
print(results_df.round(4))



=== Running XGBoost for Full dataset ===
Best params for Full: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
Performance on Full test set:
  RMSE: 5.1496
  MAE : 3.9440
  R²  : 0.5494

=== Running XGBoost for Elekta dataset ===
Best params for Elekta: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200}
Performance on Elekta test set:
  RMSE: 4.7881
  MAE : 3.5479
  R²  : 0.5822

=== Running XGBoost for Varian dataset ===
Best params for Varian: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100}
Performance on Varian test set:
  RMSE: 4.6266
  MAE : 3.2069
  R²  : 0.4989

=== XGBoost Results Summary ===
  Dataset  Best_n_estimators  Best_max_depth  Best_learning_rate  Best_CV_R2  \
0    Full                100               5                0.10      0.4943   
1  Elekta                200               6                0.05      0.4627   
2  Varian                100               6                0.10      0.3273   

     RMSE     MAE      R2  
0  5.149

In [20]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

datasets = {
    "Full":   (X_train_full, X_test_full, y_train_full, y_test_full),
    "Elekta": (X_train_E, X_test_E, y_train_E, y_test_E),
    "Varian": (X_train_V, X_test_V, y_train_V, y_test_V)
}

param_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== Decision Tree for {name} dataset ===")
    grid = GridSearchCV(
        DecisionTreeRegressor(random_state=42),
        param_grid=param_grid,
        cv=10,
        scoring="r2",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    best_params = grid.best_params_
    print(f"Best params for {name}: {best_params}")
    tree = DecisionTreeRegressor(**best_params, random_state=42)
    tree.fit(X_train, y_train)
    y_pred = tree.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"Performance on {name} test set:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE : {mae:.4f}")
    print(f"  R²  : {r2:.4f}")
    results.append({
        "Dataset": name,
        "Best_max_depth": best_params["max_depth"],
        "Best_min_samples_split": best_params["min_samples_split"],
        "Best_min_samples_leaf": best_params["min_samples_leaf"],
        "Best_CV_R2": grid.best_score_,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results)
print("\n=== Decision Tree Results Summary ===")
print(results_df.round(4))



=== Decision Tree for Full dataset ===
Best params for Full: {'max_depth': 6, 'min_samples_leaf': 4, 'min_samples_split': 10}
Performance on Full test set:
  RMSE: 6.4465
  MAE : 4.6713
  R²  : 0.2938

=== Decision Tree for Elekta dataset ===
Best params for Elekta: {'max_depth': 6, 'min_samples_leaf': 4, 'min_samples_split': 10}
Performance on Elekta test set:
  RMSE: 5.8331
  MAE : 4.3405
  R²  : 0.3799

=== Decision Tree for Varian dataset ===
Best params for Varian: {'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}
Performance on Varian test set:
  RMSE: 5.7623
  MAE : 3.9809
  R²  : 0.2227

=== Decision Tree Results Summary ===
  Dataset  Best_max_depth  Best_min_samples_split  Best_min_samples_leaf  \
0    Full               6                      10                      4   
1  Elekta               6                      10                      4   
2  Varian               2                       2                      1   

   Best_CV_R2    RMSE     MAE      R2  

In [21]:
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

datasets = {
    "Full":   (X_train_full, X_test_full, y_train_full, y_test_full),
    "Elekta": (X_train_E, X_test_E, y_train_E, y_test_E),
    "Varian": (X_train_V, X_test_V, y_train_V, y_test_V)
}

kernels = ['linear', 'poly', 'rbf']
results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== SVM Regression for {name} dataset ===")
    for kernel in kernels:
        print(f"\nKernel: {kernel}")
        svr_model = make_pipeline(
            StandardScaler(),
            SVR(kernel=kernel, C=100, gamma='auto', epsilon=0.5)
        )
        svr_model.fit(X_train, y_train)
        y_pred = svr_model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE : {mae:.4f}")
        print(f"R²  : {r2:.4f}")
        results.append({
            "Dataset": name,
            "Kernel": kernel,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

results_df = pd.DataFrame(results)
print("\n=== SVM Regression Results Summary ===")
print(results_df.round(4))



=== SVM Regression for Full dataset ===

Kernel: linear
RMSE: 6.1889
MAE : 4.6014
R²  : 0.3491

Kernel: poly
RMSE: 6.9137
MAE : 5.0182
R²  : 0.1878

Kernel: rbf
RMSE: 5.7346
MAE : 4.1702
R²  : 0.4412

=== SVM Regression for Elekta dataset ===

Kernel: linear
RMSE: 5.8910
MAE : 4.2908
R²  : 0.3675

Kernel: poly
RMSE: 6.4940
MAE : 4.5506
R²  : 0.2314

Kernel: rbf
RMSE: 5.4086
MAE : 3.7980
R²  : 0.4669

=== SVM Regression for Varian dataset ===

Kernel: linear
RMSE: 6.0353
MAE : 3.7794
R²  : 0.1473

Kernel: poly
RMSE: 6.8168
MAE : 4.2353
R²  : -0.0878

Kernel: rbf
RMSE: 4.9860
MAE : 3.2724
R²  : 0.4180

=== SVM Regression Results Summary ===
  Dataset  Kernel    RMSE     MAE      R2
0    Full  linear  6.1889  4.6014  0.3491
1    Full    poly  6.9137  5.0182  0.1878
2    Full     rbf  5.7346  4.1702  0.4412
3  Elekta  linear  5.8910  4.2908  0.3675
4  Elekta    poly  6.4940  4.5506  0.2314
5  Elekta     rbf  5.4086  3.7980  0.4669
6  Varian  linear  6.0353  3.7794  0.1473
7  Varian    pol

In [22]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

def logit(p):
    return np.log(p / (1 - p))

def inv_logit(z):
    return 1 / (1 + np.exp(-z))

datasets = {
    "Full":   (X_train_full, X_test_full, y_train_full, y_test_full),
    "Elekta": (X_train_E, X_test_E, y_train_E, y_test_E),
    "Varian": (X_train_V, X_test_V, y_train_V, y_test_V)
}

models = {
    "Lasso": Lasso(alpha=0.1, max_iter=10000),
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
}

results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== Logit Regression for {name} dataset ===")
    
    eps = 1e-6
    y_train_s = np.clip(y_train / 100, eps, 1 - eps)
    y_test_s = np.clip(y_test / 100, eps, 1 - eps)
    y_train_logit = logit(y_train_s)
    
    for model_name, model in models.items():
        print(f"\nModel: {model_name}")
        pipe = make_pipeline(StandardScaler(), model)
        pipe.fit(X_train, y_train_logit)
        y_pred_logit = pipe.predict(X_test)
        y_pred = inv_logit(y_pred_logit) * 100
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE : {mae:.4f}")
        print(f"R²  : {r2:.4f}")
        results.append({
            "Dataset": name,
            "Model": model_name,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

results_df = pd.DataFrame(results)
print("\n=== Logit-Transformed Regression Results Summary ===")
print(results_df.round(4))



=== Logit Regression for Full dataset ===

Model: Lasso
RMSE: 6.8918
MAE : 5.0080
R²  : 0.1929

Model: Ridge
RMSE: 6.8078
MAE : 4.8665
R²  : 0.2125

Model: ElasticNet
RMSE: 6.7584
MAE : 4.9053
R²  : 0.2238

=== Logit Regression for Elekta dataset ===

Model: Lasso
RMSE: 6.3641
MAE : 4.4506
R²  : 0.2618

Model: Ridge
RMSE: 6.4563
MAE : 4.4663
R²  : 0.2403

Model: ElasticNet
RMSE: 6.2801
MAE : 4.3788
R²  : 0.2812

=== Logit Regression for Varian dataset ===

Model: Lasso
RMSE: 6.3415
MAE : 3.9044
R²  : 0.0585

Model: Ridge
RMSE: 6.6544
MAE : 4.2200
R²  : -0.0366

Model: ElasticNet
RMSE: 6.2203
MAE : 3.8968
R²  : 0.0942

=== Logit-Transformed Regression Results Summary ===
  Dataset       Model    RMSE     MAE      R2
0    Full       Lasso  6.8918  5.0080  0.1929
1    Full       Ridge  6.8078  4.8665  0.2125
2    Full  ElasticNet  6.7584  4.9053  0.2238
3  Elekta       Lasso  6.3641  4.4506  0.2618
4  Elekta       Ridge  6.4563  4.4663  0.2403
5  Elekta  ElasticNet  6.2801  4.3788  0.281